<a href="https://colab.research.google.com/github/AniShriyak/Bio-Inspired-Lab/blob/main/BIS_Ant_colony.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# ---------------------------------------------------------
# 1. Power System Problem Configuration
# ---------------------------------------------------------
# Cost coefficients: C_i = a_i * P_i^2 + b_i * P_i + c_i
# Columns: [a, b, c, P_min, P_max]
generators = np.array([
    [0.00156, 7.92, 561, 100, 600],  # Generator 1
    [0.00194, 7.85, 310, 100, 400],  # Generator 2
    [0.00482, 7.97, 78,   50, 200]   # Generator 3
])

P_demand = 850.0  # Total demand in MW

# Extract coefficients and bounds
a = generators[:, 0]
b = generators[:, 1]
c = generators[:, 2]
P_min = generators[:, 3]
P_max = generators[:, 4]
n_units = len(generators)

# Penalty multiplier for power balance constraint violations
PENALTY_FACTOR = 1e5

def fitness_function(P):
    """
    Computes total cost + penalty for power balance mismatch.
    P shape: (swarm_size, n_units)
    """
    # Calculate generation costs for all particles
    fuel_cost = np.sum(a * (P**2) + b * P + c, axis=1)

    # Power balance error: Sum(P_i) - P_demand
    mismatch = np.abs(np.sum(P, axis=1) - P_demand)

    # Total fitness = fuel cost + penalty
    total_cost = fuel_cost + PENALTY_FACTOR * (mismatch ** 2)
    return total_cost

# ---------------------------------------------------------
# 2. PSO Hyperparameters
# ---------------------------------------------------------
swarm_size = 50
max_iter = 200
w = 0.7        # Inertia weight
c1 = 1.5       # Cognitive coefficient
c2 = 1.5       # Social coefficient

# Velocity limits (10% of generator operating range)
V_min = -0.1 * (P_max - P_min)
V_max = 0.1 * (P_max - P_min)

# ---------------------------------------------------------
# 3. Swarm Initialization
# ---------------------------------------------------------
# Random initial positions bounded by generator limits
X = np.random.uniform(low=P_min, high=P_max, size=(swarm_size, n_units))
V = np.random.uniform(low=V_min, high=V_max, size=(swarm_size, n_units))

pBest = np.copy(X)
pBest_fit = fitness_function(X)

gBest_idx = np.argmin(pBest_fit)
gBest = np.copy(pBest[gBest_idx])
gBest_fit = pBest_fit[gBest_idx]

# ---------------------------------------------------------
# 4. Optimization Loop
# ---------------------------------------------------------
for iteration in range(max_iter):
    # Dynamic inertia weight decay (improves convergence)
    w_curr = w * (1 - 0.5 * (iteration / max_iter))

    r1 = np.random.rand(swarm_size, n_units)
    r2 = np.random.rand(swarm_size, n_units)

    # Velocity Update
    V = (w_curr * V) + (c1 * r1 * (pBest - X)) + (c2 * r2 * (gBest - X))
    V = np.clip(V, V_min, V_max)

    # Position Update
    X = X + V
    X = np.clip(X, P_min, P_max)  # Enforce capacity limits

    # Evaluate Fitness
    fit = fitness_function(X)

    # Update Personal Best
    better_mask = fit < pBest_fit
    pBest[better_mask] = X[better_mask]
    pBest_fit[better_mask] = fit[better_mask]

    # Update Global Best
    min_fit_idx = np.argmin(pBest_fit)
    if pBest_fit[min_fit_idx] < gBest_fit:
        gBest = np.copy(pBest[min_fit_idx])
        gBest_fit = pBest_fit[min_fit_idx]

# ---------------------------------------------------------
# 5. Results Output
# ---------------------------------------------------------
total_generation = np.sum(gBest)
fuel_cost_only = np.sum(a * (gBest**2) + b * gBest + c)

print("--- Economic Load Dispatch Solution (PSO) ---")
for i in range(n_units):
    print(f"Generator {i+1}: {gBest[i]:.2f} MW (Limits: [{P_min[i]}, {P_max[i]}])")

print(f"\nTotal Power Generated: {total_generation:.2f} MW (Target: {P_demand} MW)")
print(f"Power Balance Error:   {abs(total_generation - P_demand):.4f} MW")
print(f"Optimal Fuel Cost:     ${fuel_cost_only:.2f}/hr")

--- Economic Load Dispatch Solution (PSO) ---
Generator 1: 308.70 MW (Limits: [100.0, 600.0])
Generator 2: 375.43 MW (Limits: [100.0, 400.0])
Generator 3: 165.86 MW (Limits: [50.0, 200.0])

Total Power Generated: 850.00 MW (Target: 850.0 MW)
Power Balance Error:   0.0004 MW
Optimal Fuel Cost:     $8217.72/hr
